# Joinability Shortlist — per fonte

**Obiettivo**: per ogni fonte monitorata, produrre una shortlist dei dataset
con chiavi di join riconosciute, per capire subito quali dati possiamo unire
al catalogo esistente.

**Non** un catalog score astratto — un elenco operativo di "questi dataset
hanno colonne che possono agganciarsi ad altri."

**Data**: 2026-06-08
**Fonte**: source_check_results.parquet da S3 (CI post-PR #331)

In [1]:
import json
from collections import Counter

from lab_connectors.duckdb import gcs_connect

S3_SCR = "s3://dataciviclab-clean/catalog_inventory/source-check/source_check_results.parquet"

with gcs_connect(S3_SCR) as con:
    df = con.execute("SELECT * FROM read_parquet(?)", [S3_SCR]).fetchdf()

print(f"Source-check: {len(df)} items, {df['source_id'].nunique()} sources")
print(f"Check piu recente: {df['check_timestamp'].max()}")

Source-check: 9771 items, 24 sources
Check piu recente: 2026-06-08 14:20:26+01:00


---
## 1. Panoramica join_keys

In [2]:
has_join = df[df["join_keys"].notna()].copy()
has_join["n_keys"] = has_join["join_keys"].apply(
    lambda x: len(json.loads(x)) if isinstance(x, str) else 0
)

print(f"Item totali: {len(df)}")
print(f"Con colonne profilate: {df['columns'].notna().sum()}")
print(f"Con join_keys: {len(has_join)} ({len(has_join) / len(df) * 100:.1f}%)")
print(f"Con joinability_score > 0: {(df['joinability_score'] > 0).sum()}")
print()
print("--- Per fonte ---")
summary = (
    has_join.groupby("source_id")
    .agg(
        n_with_keys=("join_keys", "count"),
        mean_score=("joinability_score", "mean"),
        max_score=("joinability_score", "max"),
        total_cols=("columns", lambda x: x.notna().sum()),
    )
    .sort_values("n_with_keys", ascending=False)
)
summary["pct_with_keys"] = (summary["n_with_keys"] / summary["total_cols"] * 100).round(1)
print(summary.to_string())

Item totali: 9771
Con colonne profilate: 2118
Con join_keys: 1009 (10.3%)
Con joinability_score > 0: 1009

--- Per fonte ---
                   n_with_keys  mean_score  max_score  total_cols  pct_with_keys
source_id                                                                       
mim_opendata               371   29.509434       43.0         371          100.0
istat_sdmx                 254   14.724409       46.0           0            inf
openga                     200   16.125000       30.0         200          100.0
unioncamere                 86   26.116279       50.0          86          100.0
inps                        66   23.000000       50.0          66          100.0
ministero_interno           20   35.250000       73.0          20          100.0
mit_opendata                12   31.083333       70.0          12          100.0


In [3]:
# Distribuzione globale delle chiavi
all_keys = Counter()
for _, r in has_join.iterrows():
    for k in json.loads(r["join_keys"]):
        all_keys[k] += 1

print("--- Distribuzione chiavi (globale) ---")
for k, c in all_keys.most_common():
    print(f"  {k:<20s} {c:>4} item ({c / len(has_join) * 100:.0f}%)")
print()
print(f"Tipi di chiave diversi: {len(all_keys)}")

--- Distribuzione chiavi (globale) ---
  anno                  754 item (75%)
  codice_scuola         321 item (32%)
  provincia             168 item (17%)
  istat_comune           95 item (9%)
  eta                    63 item (6%)
  sesso                  49 item (5%)
  cittadinanza           47 item (5%)
  istat_regione          42 item (4%)
  mese                   37 item (4%)

Tipi di chiave diversi: 9


---
## 2. Shortlist per fonte

Per ogni fonte, i top 5 dataset per joinability_score, con le chiavi trovate.

In [4]:
for sid in has_join["source_id"].value_counts().index:
    sub = has_join[has_join["source_id"] == sid].sort_values("joinability_score", ascending=False)
    print(f"\n{'=' * 70}")
    print(f"  {sid} — {len(sub)} item con chiavi")
    print(f"{'=' * 70}")

    # Distributione chiavi per questa fonte
    src_keys = Counter()
    for _, r in sub.iterrows():
        for k in json.loads(r["join_keys"]):
            src_keys[k] += 1
    key_str = ", ".join(f"{k}={c}" for k, c in src_keys.most_common())
    print(f"  Chiavi: {key_str}")
    print()

    # Top 5
    top = sub.head(5)
    for i, (_, r) in enumerate(top.iterrows(), 1):
        title = str(r.get("title", r.get("item_name", "")))[:65]
        keys = list(json.loads(r["join_keys"]).keys())
        score = r["joinability_score"]
        intake = r.get("intake_score", 0)
        fmt = str(r.get("resource_format", ""))[:6]
        print(f"  {i}. {title:<65s}")
        print(f"     keys={keys}  score={score:.0f}  intake={intake:.0f}  fmt={fmt}")


  mim_opendata — 371 item con chiavi
  Chiavi: anno=351, codice_scuola=321, provincia=84, cittadinanza=35



  1. SCUANAAU istruzione                                              
     keys=['anno', 'provincia', 'codice_scuola']  score=43  intake=82  fmt=CSV
  2. SCUANAAU istruzione                                              
     keys=['anno', 'provincia', 'codice_scuola']  score=43  intake=82  fmt=CSV
  3. SCUANAAU istruzione                                              
     keys=['anno', 'provincia', 'codice_scuola']  score=43  intake=82  fmt=CSV
  4. SCUANAAU istruzione                                              
     keys=['anno', 'provincia', 'codice_scuola']  score=43  intake=82  fmt=CSV
  5. SCUANAAU istruzione                                              
     keys=['anno', 'provincia', 'codice_scuola']  score=43  intake=82  fmt=CSV

  istat_sdmx — 254 item con chiavi
  Chiavi: anno=96, eta=63, istat_regione=33, sesso=30, istat_comune=29, mese=22, provincia=16, cittadinanza=9

  1. Resident population by age groups (five-year) and gender - munici
     keys=['istat_comune', 'sess

  Chiavi: istat_regione=5, anno=5, istat_comune=4, provincia=4, mese=1

  1. Uffici della Motorizzazione                                      
     keys=['istat_comune', 'istat_regione', 'provincia']  score=70  intake=57  fmt=CSV
  2. Storico interventi previsti dai Contratti di Programma RFI e ANAS
     keys=['istat_comune', 'istat_regione', 'provincia']  score=70  intake=100  fmt=CSV
  3. Incidenti per tipo di veicoli coinvolti e Regione - Anni 2001-201
     keys=['istat_regione', 'anno']  score=40  intake=84  fmt=CSV
  4. Incidentalità stradale - Anno 2014                               
     keys=['istat_regione', 'provincia']  score=35  intake=82  fmt=CSV
  5. Posti Barca                                                      
     keys=['istat_comune']  score=30  intake=100  fmt=CSV


---
## 3. Cross-reference con catalogo esistente

Usa la logica di `joinability_scan.py` per trovare quali dataset del catalogo
esistente diventerebbero joinabili.

In [5]:
import re

from lab_connectors.http import HttpClient

# Carica clean_catalog.json
client = HttpClient(timeout=30)
result = client.get(
    "https://raw.githubusercontent.com/dataciviclab/dataset-incubator/main/registry/clean_catalog.json"
)
catalog = result.response.json() if result.is_ok else []
datasets = catalog if isinstance(catalog, list) else catalog.get("datasets", [])
client.close()

# Indicizza
catalog_idx = {}
for ds in datasets:
    slug = ds.get("slug", ds.get("name", ""))
    cols = ds.get("columns", [])
    col_names = (
        [c.get("name", str(c)) if isinstance(c, dict) else str(c) for c in cols]
        if isinstance(cols, list)
        else []
    )
    catalog_idx[slug] = {
        "slug": slug,
        "name": ds.get("name", slug),
        "columns": col_names,
        "col_set": set(c.lower() for c in col_names),
    }

# Bridge table
BRIDGE_SLUG = "bdap_anagrafe_enti"
bridge = catalog_idx.get(BRIDGE_SLUG)
bridge_keys = set()
if bridge:
    KEY_PATTERNS = [  # subset di joinability_scan.py
        (
            "istat_comune",
            r"(?i)(codice_istat_comune|codice_comune_istat|^codice_comune$|^pro_com$|^comune$)",
        ),
        ("istat_regione", r"(?i)(codice_istat_regione|^codice_regione$|^codreg$)"),
        ("provincia", r"(?i)(sigla_provincia|^provincia$|codice_provincia)"),
        ("codice_catastale", r"(?i)(codice_catastale|cod_catastale)"),
        ("codice_ente", r"(?i)(codice_ente_ipa|^id_ente$|codice_ente_siope)"),
        ("codice_scuola", r"(?i)(codice_scuola|codicescuola)"),
        ("ateco", r"(?i)(codice_ateco|^ateco$)"),
    ]
    for col in bridge["col_set"]:
        for key_name, pattern in KEY_PATTERNS:
            if re.search(pattern, col):
                bridge_keys.add(key_name)
    print(f"Bridge `{BRIDGE_SLUG}` -> {len(bridge_keys)} chiavi: {sorted(bridge_keys)}")
else:
    print(f"Bridge `{BRIDGE_SLUG}` non trovato nel catalogo")

print(f"\nCatalogo: {len(catalog_idx)} dataset")

Bridge `bdap_anagrafe_enti` -> 6 chiavi: ['ateco', 'codice_catastale', 'codice_ente', 'istat_comune', 'istat_regione', 'provincia']

Catalogo: 39 dataset


In [6]:
def cross_ref_item(found_keys, catalog_index, bridge_semantic_keys):
    """Trova dataset nel catalogo joinabili con questo item."""
    our_cols = set()
    for cols in found_keys.values():
        for c in cols:
            our_cols.add(c.lower())
    semantic_keys = set(found_keys.keys())
    bridge_hit = bool(semantic_keys & bridge_semantic_keys)

    matches = []
    for slug, info in catalog_index.items():
        direct = our_cols & info["col_set"]
        if direct:
            matches.append({"slug": slug, "name": info["name"], "type": "diretto"})
            continue
        if bridge_hit:
            ds_semantic = set()
            for col in info["col_set"]:
                for key_name, pattern in KEY_PATTERNS:
                    if re.search(pattern, col):
                        ds_semantic.add(key_name)
            if ds_semantic & bridge_semantic_keys:
                matches.append({"slug": slug, "name": info["name"], "type": "via bridge"})
    return matches


# Cross-ref per ogni item con join_keys (solo top 3 per fonte per velocita)
print("--- Item con match nel catalogo ---")
for sid in has_join["source_id"].value_counts().index[:6]:  # prime 6 fonti
    sub = (
        has_join[has_join["source_id"] == sid]
        .sort_values("joinability_score", ascending=False)
        .head(3)
    )
    for _, r in sub.iterrows():
        found = json.loads(r["join_keys"])
        matches = cross_ref_item(found, catalog_idx, bridge_keys)
        if matches:
            title = str(r.get("title", ""))[:50]
            print(f"\n  [{sid}] {title}")
            print(f"         keys={list(found.keys())}")
            for m in matches[:4]:
                print(f"         -> {m['slug']:<25s} ({m['type']})")

--- Item con match nel catalogo ---

  [mim_opendata] SCUANAAU istruzione
         keys=['anno', 'provincia', 'codice_scuola']
         -> aifa_spesa_consumo        (via bridge)
         -> bdap_anagrafe_enti        (via bridge)
         -> bdap_lea                  (via bridge)
         -> consip_consumi_convenzione (via bridge)

  [mim_opendata] SCUANAAU istruzione
         keys=['anno', 'provincia', 'codice_scuola']
         -> aifa_spesa_consumo        (via bridge)
         -> bdap_anagrafe_enti        (via bridge)
         -> bdap_lea                  (via bridge)
         -> consip_consumi_convenzione (via bridge)

  [mim_opendata] SCUANAAU istruzione
         keys=['anno', 'provincia', 'codice_scuola']
         -> aifa_spesa_consumo        (via bridge)
         -> bdap_anagrafe_enti        (via bridge)
         -> bdap_lea                  (via bridge)
         -> consip_consumi_convenzione (via bridge)

  [istat_sdmx] Resident population by age groups (five-year) and 
         

---
## 4. Sintesi operativa

Dataset che potremmo portare subito nel Lab, ordinati per fonte e joinabilità.

In [7]:
print("=== SHORTLIST OPERATIVA (top 50 item per joinability) ===\n")
print(f"{'#':<3s} {'Fonte':<18s} {'Dataset':<55s} {'Chiavi':<30s} {'JScore':>6s} {'Intake':>6s}")
print("-" * 120)

top50 = has_join.sort_values("joinability_score", ascending=False).head(50)
for i, (_, r) in enumerate(top50.iterrows(), 1):
    title = str(r.get("title", r.get("item_name", "")))[:52]
    keys = ",".join(json.loads(r["join_keys"]).keys())[:28]
    print(
        f"{i:<3d} {r['source_id']:<18s} {title:<55s} {keys:<30s} {r['joinability_score']:>6.0f} {r.get('intake_score', 0):>6.0f}"
    )

=== SHORTLIST OPERATIVA (top 50 item per joinability) ===

#   Fonte              Dataset                                                 Chiavi                         JScore Intake
------------------------------------------------------------------------------------------------------------------------


1   ministero_interno  Elezioni Comunali 2024                                  istat_comune,istat_regione,p       73     82
2   ministero_interno  Elezioni Regionali 2023                                 istat_comune,istat_regione,p       73     82
3   mit_opendata       Storico interventi previsti dai Contratti di Program    istat_comune,istat_regione,p       70    100
4   mit_opendata       Uffici della Motorizzazione                             istat_comune,istat_regione,p       70     57
5   ministero_interno  Attività residenti                                      istat_comune,istat_regione,p       70     92
6   ministero_interno  Popolazione residente                                   istat_comune,istat_regione,p       70    100
7   unioncamere        Torino - Imprese Femminili al IV Trimestre 2024         istat_comune,anno                  50     92
8   unioncamere        Torino - Demografia delle Imprese al III Trimestre 2    istat_comune,anno                  50     92
9   unio

---
## 5. Note

- **556 item con join_keys** su 8.586 totali (6.5%) — primo run post-PR #331.
- La copertura crescera' ogni settimana con la CI.
- `codice_scuola` domina grazie a mim_opendata (371 item con chiavi).
- `istat_comune` e `provincia` sono le chiavi geografiche piu' comuni.
- Prossimo passo: integrare `joinability_scan.py` per cross-ref completo.